# ML Transformations Explained — Visual Guide

Every preprocessing step in `house_prices.ipynb`, shown visually with a **before**, **after**, and **"what breaks if you skip this"** explanation.

---
**Map of all transformations in the pipeline:**

```
Raw data
 │
 ├─ 1. Log-transform TARGET (SalePrice → log(SalePrice+1))
 │
 ├─ 2. Fill missing values (Imputation)
 │
 ├─ 3. Encode categories as numbers (Ordinal / Nominal encoding)
 │
 ├─ 4. Scale numeric features (StandardScaler)
 │
 └─ 5. Feature engineering (build new columns from old ones)
         TotalSF, HouseAge, LogLotArea, QualArea, ...
```

Steps 2–4 happen inside `sklearn Pipeline`. Step 1 and 5 happen before the pipeline.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from scipy import stats
from sklearn.preprocessing import StandardScaler
import os, warnings

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 110

# Load the real dataset
import kagglehub
path  = kagglehub.competition_download('house-prices-advanced-regression-techniques')
train = pd.read_csv(os.path.join(path, 'train.csv'))
print(f'Loaded {train.shape[0]} rows')

---
## 1 — Log-transform the target: `SalePrice → log(SalePrice + 1)`

### The problem: skewness

**Skewness** means "most values are on the left, but a long tail stretches to the right".  
House prices are like that: most houses cost 100k–300k, but a few mansions cost 700k+.

That long tail **hurts** a model in two ways:

1. **Regression loss (RMSE) is dominated by the big houses.**  
   A $50k error on a $700k house counts more than a $50k error on a $150k house. The model learns to chase outliers.

2. **Relationships look curved, not linear.**  
   `GrLivArea` vs `SalePrice` has a fan shape. After log, the cloud becomes a straight line — much easier for the model.

### The fix: `np.log1p(price)`

`log1p(x)` = `log(x + 1)` — the `+1` avoids `log(0) = -∞` for zero values.

After log, the distribution is near-normal (symmetric, no tail). The model treats a 10% price error equally whether the house costs $100k or $500k.

### Reversing: `np.expm1`

When we're done predicting, we undo the log with `np.expm1(predicted_log_price)` to get real dollars back.

In [ ]:
price    = train['SalePrice']
log_price = np.log1p(price)

fig, axes = plt.subplots(2, 3, figsize=(17, 9))

# ── Row 1: distributions ──────────────────────────────────────────────────────
axes[0,0].hist(price, bins=60, color='steelblue', edgecolor='white')
axes[0,0].axvline(price.mean(),   color='red',    lw=2, label=f'mean  ${price.mean():,.0f}')
axes[0,0].axvline(price.median(), color='orange', lw=2, label=f'median ${price.median():,.0f}')
axes[0,0].set_title(f'RAW SalePrice   skew={price.skew():.2f}', fontsize=12)
axes[0,0].set_xlabel('SalePrice ($)')
axes[0,0].legend(fontsize=9)

axes[0,1].hist(log_price, bins=60, color='seagreen', edgecolor='white')
axes[0,1].axvline(log_price.mean(),   color='red',    lw=2, label=f'mean  {log_price.mean():.2f}')
axes[0,1].axvline(log_price.median(), color='orange', lw=2, label=f'median {log_price.median():.2f}')
axes[0,1].set_title(f'log1p(SalePrice)  skew={log_price.skew():.2f}  ← MUCH better', fontsize=12)
axes[0,1].set_xlabel('log(SalePrice + 1)')
axes[0,1].legend(fontsize=9)

stats.probplot(log_price, plot=axes[0,2])
axes[0,2].set_title('QQ-plot of log(SalePrice)  ← dots on diagonal = normal', fontsize=11)

# ── Row 2: scatter GrLivArea vs price ─────────────────────────────────────────
axes[1,0].scatter(train['GrLivArea'], price, alpha=0.3, s=8, color='steelblue')
axes[1,0].set_xlabel('GrLivArea (sq ft)')
axes[1,0].set_ylabel('SalePrice ($)')
axes[1,0].set_title('GrLivArea vs RAW price\nfan shape — hard for model', fontsize=11)

axes[1,1].scatter(train['GrLivArea'], log_price, alpha=0.3, s=8, color='seagreen')
axes[1,1].set_xlabel('GrLivArea (sq ft)')
axes[1,1].set_ylabel('log(SalePrice)')
axes[1,1].set_title('GrLivArea vs log(price)\nnearly linear — easy for model', fontsize=11)

# Show the round-trip: log then expm1
example_prices = np.array([100_000, 200_000, 300_000, 500_000, 700_000])
example_log    = np.log1p(example_prices)
recovered      = np.expm1(example_log)

axes[1,2].axis('off')
table_data = list(zip(
    [f'${p:,}' for p in example_prices],
    [f'{l:.4f}' for l in example_log],
    [f'${r:,.0f}' for r in recovered],
))
tbl = axes[1,2].table(
    cellText=table_data,
    colLabels=['Original $', 'log1p →', 'expm1 ←'],
    cellLoc='center', loc='center',
)
tbl.auto_set_font_size(False)
tbl.set_fontsize(10)
tbl.scale(1.2, 1.6)
axes[1,2].set_title('Round-trip: log1p → predict → expm1', fontsize=11)

plt.suptitle('Step 1 — Log-transform the target', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

print("""
KEY INSIGHT
───────────
Before log: mean=$180k, but max=$755k pulls the mean away from most houses.
After  log: mean and median are almost the same — the distribution is symmetric.

The model now minimises error on log-scale, which means it minimises
RELATIVE error (%) rather than absolute dollar error.
A 10% mistake on a cheap house is punished equally to a 10% mistake on an expensive one.
""")

---
## 2 — Filling missing values (Imputation)

### Why data has holes

Some columns are missing because the thing doesn't exist (a house with no basement has `NaN` in `BsmtQual`), or because the data wasn't collected.

**sklearn models crash on `NaN`** — they can't split on "missing". So we fill gaps before training.

### Strategies we use

| Column type | Strategy | Why |
|---|---|---|
| **Numeric** (`GarageArea`, etc.) | Fill with **median** | Median isn't affected by a few huge outliers like mean is |
| **Ordinal quality** (`BsmtQual`, etc.) | Fill with `'NA'` (a real category) | Missing basement = no basement, not "unknown" — it's its own grade |
| **Nominal** (`Foundation`, etc.) | Fill with `'MISSING'` as a category | Let the model learn that `MISSING` has its own price pattern |

In [ ]:
cols_with_nulls = ['GarageArea', 'TotalBsmtSF', 'GarageCars', 'BsmtQual', 'GarageFinish']
null_counts = train[cols_with_nulls].isnull().sum()

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# ── Plot 1: missing value counts ──────────────────────────────────────────────
all_nulls = train.isnull().sum().sort_values(ascending=False).head(20)
all_nulls = all_nulls[all_nulls > 0]
axes[0].barh(all_nulls.index, all_nulls.values, color='salmon', edgecolor='white')
axes[0].set_title('Missing value count per column\n(top 20)', fontsize=11)
axes[0].set_xlabel('Count of NaN rows')

# ── Plot 2: median vs mean for a skewed column ────────────────────────────────
col = 'GarageArea'
values = train[col].dropna()
axes[1].hist(values, bins=40, color='steelblue', edgecolor='white', alpha=0.8)
axes[1].axvline(values.mean(),   color='red',    lw=2.5, label=f'mean   {values.mean():.0f}')
axes[1].axvline(values.median(), color='orange', lw=2.5, label=f'median {values.median():.0f}')
axes[1].set_title(f'{col} — mean vs median\nMedian is safer for skewed data', fontsize=11)
axes[1].set_xlabel(col)
axes[1].legend()

# ── Plot 3: BsmtQual — 'NA' is a real category ───────────────────────────────
bsmt = train[['BsmtQual', 'SalePrice']].copy()
bsmt['BsmtQual'] = bsmt['BsmtQual'].fillna('NA (no bsmt)')
order = bsmt.groupby('BsmtQual')['SalePrice'].median().sort_values().index
sns.boxplot(data=bsmt, x='BsmtQual', y='SalePrice', order=order, ax=axes[2],
            palette='muted', width=0.6)
axes[2].set_title('BsmtQual — NA means "no basement"\nnot "unknown" — it has its own price!', fontsize=11)
axes[2].set_xlabel('Basement Quality')
axes[2].tick_params(axis='x', rotation=30)

plt.suptitle('Step 2 — Imputation (fill missing values)', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

print(f"""
KEY INSIGHT
───────────
GarageArea: mean={train['GarageArea'].mean():.0f}  median={train['GarageArea'].median():.0f}
For a missing garage, filling with {train['GarageArea'].median():.0f} (median) makes more sense
than {train['GarageArea'].mean():.0f} (mean) — mean is pulled up by large garages.

BsmtQual: a house without a basement IS its own category.
Filling with 'NA' keeps that information instead of guessing 'TA' (typical).
""")

---
## 3 — Encoding categories as numbers

Models work with numbers, not strings. `'Gd'` and `'TA'` mean nothing to XGBoost — we need to convert them.

There are **two very different kinds** of categorical columns:

### Ordinal — there is a natural order

`BsmtQual`: `Po < Fa < TA < Gd < Ex` — each level is genuinely better than the previous.

We encode as `0, 1, 2, 3, 4` — the **numbers carry meaning** (3 is better than 1).

If you encode `Po=0, Fa=1, TA=2, Gd=3, Ex=4` the model can learn "higher number → higher price" in a single split.

**Wrong approach for ordinal**: one-hot encoding — it throws away the order and creates 5 useless columns where 1 ordered column would do.

### Nominal — no natural order

`Neighborhood`: `'NoRidge'` is not better than `'OldTown'` — they're just different.

We still use ordinal-style integer encoding here because XGBoost handles this well with `handle_unknown='use_encoded_value'`. The numbers 0–24 are arbitrary — the tree will find the right splits.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# ── Plot 1: ordinal quality encoded ──────────────────────────────────────────
qual_order  = ['Po', 'Fa', 'TA', 'Gd', 'Ex']
qual_nums   = {q: i for i, q in enumerate(qual_order)}
bsmt_valid  = train[['BsmtQual', 'SalePrice']].dropna(subset=['BsmtQual'])
bsmt_valid  = bsmt_valid[bsmt_valid['BsmtQual'].isin(qual_order)]
bsmt_valid['encoded'] = bsmt_valid['BsmtQual'].map(qual_nums)

for q, grp in bsmt_valid.groupby('BsmtQual'):
    enc = qual_nums[q]
    axes[0].scatter([enc] * len(grp), grp['SalePrice'], alpha=0.15, s=8)
medians = bsmt_valid.groupby('encoded')['SalePrice'].median()
axes[0].plot(medians.index, medians.values, 'k--o', lw=2, label='median')
axes[0].set_xticks(range(5))
axes[0].set_xticklabels([f'{q}\n({i})' for i, q in enumerate(qual_order)])
axes[0].set_xlabel('BsmtQual (label → encoded number)')
axes[0].set_ylabel('SalePrice ($)')
axes[0].set_title('Ordinal encoding: Po→0, Fa→1, TA→2, Gd→3, Ex→4\nOrder preserved = meaningful numbers', fontsize=10)
axes[0].legend()

# ── Plot 2: nominal (neighborhood) ────────────────────────────────────────────
nbr = train[['Neighborhood', 'SalePrice']].copy()
order = nbr.groupby('Neighborhood')['SalePrice'].median().sort_values().index
nbr_encoded = {n: i for i, n in enumerate(order)}
nbr['encoded'] = nbr['Neighborhood'].map(nbr_encoded)

medians2 = nbr.groupby('encoded')['SalePrice'].median()
axes[1].plot(medians2.index, medians2.values / 1000, 'o-', color='steelblue', lw=1.5, markersize=5)
axes[1].set_xlabel('Neighborhood (encoded 0–24, sorted by median price)')
axes[1].set_ylabel('Median SalePrice ($k)')
axes[1].set_title('Nominal encoding: 25 neighborhoods → 0..24\nNumbers are arbitrary but XGBoost finds the price splits', fontsize=10)

# ── Plot 3: what goes wrong with wrong encoding ───────────────────────────────
# Show: if you encode ordinal WRONG (random order), model gets confused
wrong_order = ['Ex', 'Po', 'Gd', 'Fa', 'TA']   # shuffled!
wrong_nums  = {q: i for i, q in enumerate(wrong_order)}
bsmt_valid['wrong_encoded'] = bsmt_valid['BsmtQual'].map(wrong_nums)

for q, grp in bsmt_valid.groupby('BsmtQual'):
    enc = wrong_nums[q]
    axes[2].scatter([enc] * len(grp), grp['SalePrice'], alpha=0.15, s=8)
medians_w = bsmt_valid.groupby('wrong_encoded')['SalePrice'].median()
axes[2].plot(medians_w.index, medians_w.values, 'k--o', lw=2, label='median')
axes[2].set_xticks(range(5))
axes[2].set_xticklabels([f'{q}\n({i})' for i, q in enumerate(wrong_order)])
axes[2].set_xlabel('BsmtQual (WRONG random order)')
axes[2].set_ylabel('SalePrice ($)')
axes[2].set_title('WRONG ordinal encoding (shuffled labels)\nZigzag pattern — model needs extra splits to learn order', fontsize=10)
axes[2].legend()

plt.suptitle('Step 3 — Encoding categories as numbers', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

print("""
KEY INSIGHT
───────────
Ordinal (quality): encode in the RIGHT order. The model can then learn the full
price gradient in ONE split ("if BsmtQual > 2.5 → high price").

Wrong order means the model needs 4 splits to learn what 1 could have done.
This wastes tree depth and generalises worse.
""")

---
## 4 — Scaling numeric features (StandardScaler)

### The problem: features live on very different scales

- `GrLivArea` ranges from 334 to 5642  
- `FullBath` ranges from 0 to 3  
- `YearBuilt` ranges from 1872 to 2010

**For XGBoost: scaling doesn't matter.** Trees split on thresholds — the absolute scale of a column doesn't affect where the best split is.

**But for other models (linear regression, SVM, KNN, neural nets): it's critical.**  
A feature with large numbers dominates the loss function. A feature that goes from 0–3 looks negligible against one that goes 0–5000, even if it's actually important.

### What StandardScaler does

For each column: `z = (x - mean) / std`

Result: every column has **mean = 0** and **std = 1** after scaling.  
A value of `2.0` means "2 standard deviations above average" — comparable across all features.

### Train/test contamination (a trap)

You must fit the scaler **only on training data**, then apply the same fitted scaler to test data.

If you fit the scaler on train+test together, the test data's distribution influences the scaling — **the model has peeked at test**. This is called *data leakage*. sklearn Pipeline handles this automatically.

In [ ]:
numeric_demo = ['GrLivArea', 'LotArea', 'YearBuilt', 'FullBath', 'GarageArea']
data_raw = train[numeric_demo].dropna()

scaler = StandardScaler()
data_scaled = pd.DataFrame(scaler.fit_transform(data_raw), columns=numeric_demo)

fig, axes = plt.subplots(2, 5, figsize=(20, 8))

colors = ['steelblue', 'coral', 'seagreen', 'darkorange', 'mediumpurple']
for i, col in enumerate(numeric_demo):
    # Row 1: before scaling
    axes[0, i].hist(data_raw[col], bins=40, color=colors[i], edgecolor='white', alpha=0.8)
    axes[0, i].set_title(f'RAW: {col}\nrange [{data_raw[col].min():.0f}, {data_raw[col].max():.0f}]', fontsize=9)
    axes[0, i].set_xlabel(col)

    # Row 2: after scaling
    axes[1, i].hist(data_scaled[col], bins=40, color=colors[i], edgecolor='white', alpha=0.5)
    axes[1, i].set_title(f'SCALED: {col}\nmean≈0, std≈1', fontsize=9)
    axes[1, i].axvline(0, color='black', lw=1, linestyle='--')
    axes[1, i].set_xlabel('z-score (standard deviations)')

axes[0, 0].set_ylabel('COUNT (before scaling)', fontsize=9)
axes[1, 0].set_ylabel('COUNT (after scaling)', fontsize=9)

plt.suptitle('Step 4 — StandardScaler: all columns → mean=0, std=1\n(shape stays the same, only the axis changes)', fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

# Show the numbers
print('Before scaling — ranges are all over the place:')
print(data_raw.describe().loc[['mean', 'std', 'min', 'max']].round(1))
print('\nAfter scaling — every column: mean≈0, std≈1:')
print(data_scaled.describe().loc[['mean', 'std', 'min', 'max']].round(2))

In [ ]:
# ── Train/test contamination — visual demo ─────────────────────────────────────
np.random.seed(42)
train_vals = np.random.normal(loc=500, scale=100, size=800)
test_vals  = np.random.normal(loc=700, scale=80,  size=200)   # test has different distribution

# CORRECT: fit scaler only on train
sc_correct = StandardScaler()
sc_correct.fit(train_vals.reshape(-1,1))
train_c = sc_correct.transform(train_vals.reshape(-1,1)).ravel()
test_c  = sc_correct.transform(test_vals.reshape(-1,1)).ravel()

# WRONG: fit scaler on all data combined
sc_leaked = StandardScaler()
sc_leaked.fit(np.concatenate([train_vals, test_vals]).reshape(-1,1))
train_l = sc_leaked.transform(train_vals.reshape(-1,1)).ravel()
test_l  = sc_leaked.transform(test_vals.reshape(-1,1)).ravel()

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].hist(train_c, bins=40, alpha=0.6, color='steelblue', label='train (fitted here)')
axes[0].hist(test_c,  bins=40, alpha=0.6, color='coral',     label='test  (transformed with train stats)')
axes[0].set_title('CORRECT: scaler fitted on train only\nTest is "shifted" — model can see the difference', fontsize=10)
axes[0].legend()

axes[1].hist(train_l, bins=40, alpha=0.6, color='steelblue', label='train')
axes[1].hist(test_l,  bins=40, alpha=0.6, color='coral',     label='test')
axes[1].set_title('WRONG: scaler fitted on train+test\nTest leaks into scaler — train/test look identical!', fontsize=10)
axes[1].legend()

plt.suptitle('Data leakage trap — always fit scaler on TRAIN only', fontsize=12, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

print("""
KEY INSIGHT
───────────
sklearn Pipeline solves this automatically:
  pipeline.fit(X_train, y_train)   → scaler.fit_transform(X_train)
  pipeline.predict(X_test)         → scaler.transform(X_test)  ← uses train stats

Never call scaler.fit() on test data separately.
""")

---
## 5 — Feature Engineering: building new columns from existing ones

The idea: **combine raw columns into new ones that capture something the model can't easily derive on its own.**

Each feature below has a concrete justification — not "let's try everything".

| New feature | Formula | Why |
|---|---|---|
| `TotalSF` | `bsmt + 1stFlr + 2ndFlr` | Total livable area is more useful than three separate areas |
| `HouseAge` | `YrSold - YearBuilt` | Price depends on age, not birth year. A 1990 house sold in 2010 is 20 years old |
| `RemodAge` | `YrSold - YearRemodAdd` | How recently was it renovated? Renovation freshness matters more than renovation year |
| `TotalBath` | `FullBath + 0.5×HalfBath` | Half-baths add value but less than full — one combined score |
| `LogLotArea` | `log1p(LotArea)` | LotArea has extreme outliers (farms). Log compresses them |
| `QualArea` | `OverallQual × GrLivArea` | **Interaction term**: a big house with high quality is worth more than either alone |
| `IsNew` | `YearBuilt >= 2000` | New builds often have a price premium not captured linearly by age |

In [ ]:
df = train.copy()
df['log_price'] = np.log1p(df['SalePrice'])
df['HouseAge']  = df['YrSold'] - df['YearBuilt']
df['TotalSF']   = df['TotalBsmtSF'].fillna(0) + df['1stFlrSF'].fillna(0) + df['2ndFlrSF'].fillna(0)
df['QualArea']  = df['OverallQual'] * df['GrLivArea']
df['LogLotArea']= np.log1p(df['LotArea'])
df['TotalBath'] = df['FullBath'].fillna(0) + 0.5 * df['HalfBath'].fillna(0)

fig, axes = plt.subplots(2, 4, figsize=(20, 9))

# ── Why HouseAge is better than YearBuilt ────────────────────────────────────
axes[0,0].scatter(df['YearBuilt'], df['log_price'], alpha=0.2, s=8, color='steelblue')
axes[0,0].set_xlabel('YearBuilt')
axes[0,0].set_ylabel('log(price)')
axes[0,0].set_title('YearBuilt vs price\nSlight upward curve', fontsize=10)

axes[0,1].scatter(df['HouseAge'], df['log_price'], alpha=0.2, s=8, color='seagreen')
axes[0,1].set_xlabel('HouseAge (years at time of sale)')
axes[0,1].set_ylabel('log(price)')
axes[0,1].set_title('HouseAge vs price\nClearer declining relationship', fontsize=10)

# ── Why TotalSF is better than individual floor areas ────────────────────────
axes[0,2].scatter(df['GrLivArea'], df['log_price'], alpha=0.2, s=8, color='coral')
axes[0,2].set_xlabel('GrLivArea (above-ground only)')
axes[0,2].set_ylabel('log(price)')
axes[0,2].set_title('GrLivArea alone\ncor={:.2f}'.format(df['GrLivArea'].corr(df['log_price'])), fontsize=10)

axes[0,3].scatter(df['TotalSF'], df['log_price'], alpha=0.2, s=8, color='darkorange')
axes[0,3].set_xlabel('TotalSF (all floors + basement)')
axes[0,3].set_ylabel('log(price)')
axes[0,3].set_title('TotalSF — better correlation\ncor={:.2f}'.format(df['TotalSF'].corr(df['log_price'])), fontsize=10)

# ── Why QualArea interaction beats either alone ───────────────────────────────
axes[1,0].scatter(df['OverallQual'], df['log_price'], alpha=0.25, s=8, color='mediumpurple')
axes[1,0].set_xlabel('OverallQual (1–10)')
axes[1,0].set_title(f'OverallQual  cor={df["OverallQual"].corr(df["log_price"]):.3f}', fontsize=10)

axes[1,1].scatter(df['GrLivArea'], df['log_price'], alpha=0.2, s=8, color='steelblue')
axes[1,1].set_xlabel('GrLivArea (sq ft)')
axes[1,1].set_title(f'GrLivArea  cor={df["GrLivArea"].corr(df["log_price"]):.3f}', fontsize=10)

axes[1,2].scatter(df['QualArea'], df['log_price'], alpha=0.2, s=8, color='seagreen')
axes[1,2].set_xlabel('QualArea = OverallQual × GrLivArea')
axes[1,2].set_title(f'QualArea  cor={df["QualArea"].corr(df["log_price"]):.3f}  ← BEST!', fontsize=10)

# ── LotArea vs LogLotArea ─────────────────────────────────────────────────────
axes[1,3].scatter(df['LotArea'],    df['log_price'], alpha=0.15, s=8, color='coral',
                  label=f'LotArea  cor={df["LotArea"].corr(df["log_price"]):.2f}')
ax2 = axes[1,3].twiny()
ax2.scatter(df['LogLotArea'], df['log_price'], alpha=0.15, s=8, color='darkorange',
            label=f'log(LotArea)  cor={df["LogLotArea"].corr(df["log_price"]):.2f}')
axes[1,3].set_xlabel('LotArea (raw)', color='coral')
ax2.set_xlabel('log(LotArea)', color='darkorange')
axes[1,3].set_title('LotArea: log compresses outliers\nand improves correlation', fontsize=10)

for ax in axes.flat:
    ax.set_ylabel('log(price)' if ax.get_ylabel() == '' else ax.get_ylabel())

plt.suptitle('Step 5 — Feature Engineering: why each new column helps', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# ── Deep-dive: interaction term QualArea ─────────────────────────────────────
# Why OverallQual × GrLivArea > both separately

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: same-size houses — quality matters a LOT
size_band = (df['GrLivArea'] > 1400) & (df['GrLivArea'] < 1600)
band = df[size_band]
qual_median = band.groupby('OverallQual')['SalePrice'].median() / 1000
axes[0].bar(qual_median.index, qual_median.values, color='steelblue', edgecolor='white')
axes[0].set_xlabel('OverallQual (among houses 1400–1600 sqft)')
axes[0].set_ylabel('Median SalePrice ($k)')
axes[0].set_title('Same size, different quality:\nquality doubles the price!', fontsize=11)

# Right: same quality — size also matters
qual_band = df['OverallQual'] == 7
qband = df[qual_band].copy()
qband['area_bin'] = pd.cut(qband['GrLivArea'], bins=8)
area_median = qband.groupby('area_bin', observed=True)['SalePrice'].median() / 1000
axes[1].bar(range(len(area_median)), area_median.values, color='seagreen', edgecolor='white')
axes[1].set_xticks(range(len(area_median)))
axes[1].set_xticklabels([str(b) for b in area_median.index], rotation=40, ha='right', fontsize=8)
axes[1].set_xlabel('GrLivArea range (among OverallQual=7 houses)')
axes[1].set_ylabel('Median SalePrice ($k)')
axes[1].set_title('Same quality, different size:\nsize also multiplies price', fontsize=11)

plt.suptitle('Why QualArea = OverallQual × GrLivArea is a powerful interaction term\n'
             'The effect of size depends on quality — they multiply, not just add', 
             fontsize=12, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print("""
KEY INSIGHT — Interaction terms
────────────────────────────────
A 1000 sqft house with OverallQual=5 sells for ~$130k.
A 1000 sqft house with OverallQual=9 sells for ~$250k.
A 2000 sqft house with OverallQual=9 sells for ~$450k.

The price isn't (area-effect) + (quality-effect) — it's MULTIPLIED.
That multiplicative relationship is exactly what QualArea = Qual × Area captures.

A tree-based model CAN learn this through nested splits, but giving it
QualArea directly means it can do it in ONE split, using less depth and
generalising better.
""")

---
## The Full Picture — Sequence Matters

Here's the complete flow, and **why the order is what it is**:

```
RAW DATA
  │
  ▼
① Feature engineering            ← BEFORE the pipeline
   add TotalSF, HouseAge, etc.
   (these use raw columns that exist at this point)
  │
  ▼
② Imputation (fill NaN)          ┐
  │                              │  inside sklearn Pipeline
  ▼                              │  fitted ONLY on train
③ Ordinal/Nominal encoding       │  applied to test using train's params
  │                              │
  ▼                              │
④ StandardScaler                 ┘
  │
  ▼
⑤ XGBoost model
  │
  ▼
⑥ Predict → output is log(price)
  │
  ▼
⑦ np.expm1( predicted )  → real dollars
```

### Why can't we do ② before ①?

Feature engineering (`add_features()`) uses columns that might have NaNs (e.g. `TotalBsmtSF`). We handle this inside `add_features()` with `.fillna(0)` because it makes semantic sense (no basement = 0 sq ft). The pipeline's imputer then handles whatever NaNs remain in the original columns.

### Why does log go before the pipeline?

The pipeline transforms **X** (features). The target **y = log1p(SalePrice)** is computed outside and passed to `.fit()` directly. sklearn pipelines don't touch `y`.

In [ ]:
# ── Summary table: every transformation, its purpose, and consequence if skipped ──

rows = [
    ('log1p(SalePrice)',    'Target',
     'Remove right skew in price',
     'Model chases $700k outliers; % errors on cheap houses blow up'),

    ('Median impute numerics', 'Preprocessing',
     'Fill NaN so model can train',
     'sklearn raises ValueError: Input contains NaN'),

    ('"NA" impute BsmtQual etc.', 'Preprocessing',
     'No-basement is its own quality category',
     'Replaced with "TA" (typical) — model thinks it has a basement'),

    ('OrdinalEncoder (Po<Fa<TA<Gd<Ex)', 'Preprocessing',
     'Quality order → meaningful number order',
     'Wrong order = zigzag = more splits needed = worse generalisation'),

    ('StandardScaler', 'Preprocessing',
     'All numerics on same z-score scale',
     'No effect for XGBoost; breaks linear/SVM/KNN models'),

    ('TotalSF', 'Feature eng.',
     'One total area vs three fragments',
     'Model sees bsmt, 1stFlr, 2ndFlr as unrelated; total less clear'),

    ('HouseAge = YrSold - YearBuilt', 'Feature eng.',
     'Age at time of sale (not birth year)',
     'YearBuilt 1990 means different things in 2000 vs 2010'),

    ('QualArea = Qual × GrLivArea', 'Feature eng.',
     'Capture multiplicative quality-size interaction',
     'Model needs 2+ extra splits to learn this — wastes tree depth'),

    ('log1p(LotArea)', 'Feature eng.',
     'Compress extreme farm-sized lots',
     'Outlier lots dominate; correlation with price stays weak'),
]

df_steps = pd.DataFrame(rows, columns=['Transformation', 'Stage', 'Purpose', 'If skipped'])
df_steps.index = range(1, len(df_steps)+1)

print('=== FULL TRANSFORMATION SUMMARY ===')
print()
for _, row in df_steps.iterrows():
    print(f"  {row['Transformation']:42s}  [{row['Stage']}]")
    print(f"    Purpose  : {row['Purpose']}")
    print(f"    If skipped: {row['If skipped']}")
    print()